# C10-cnn-transfer — CNNs, Transfer Learning, and CNN Training

This is the index notebook for the four-session, **345-minute** unit.
The three earlier sessions build and inspect convolutional feature extractors;
Session 4 carries C8's loss/autograd/optimizer lifecycle into a tiny CNN and
then fine-tunes only a selected parameter subset.

## Unit goals

By the end of this unit you can:

- compute 1-D and 2-D convolution in component form and use
  $\lfloor(n+2p-K)/s\rfloor+1$ independently on both spatial axes;
- trace `(N,C,H,W)` through convolution, pooling, adaptive pooling, flattening,
  and a classifier head before running the model;
- interpret feature maps, receptive fields, and the edges → textures → parts →
  objects hierarchy;
- read ResNet-50's stem/stages/bottleneck blocks, truncate at valid boundaries,
  and count parameters by hand;
- freeze by qualified-name prefix and audit both parameter-tensor membership
  and scalar counts;
- construct a convolutional classifier and train it with `CrossEntropyLoss`
  and the exact `zero_grad → forward → loss → backward → step` lifecycle;
- distinguish optimizer-owned parameters, gradients, parameter movement, and
  BatchNorm buffers, including `train()`/`eval()` and dropout behavior; and
- fine-tune selectively by freezing **before** optimizer construction, then
  rebuilding the optimizer when another layer is deliberately unfrozen.

## Prerequisites and forward path

**C7-pytorch** supplies tensors, `nn.Module`, parameters, `requires_grad`,
module inspection, and parameter counting.
**C8-neural-training** supplies stable softmax/cross-entropy, autograd,
optimizers, the training lifecycle, BatchNorm/dropout, and robust training
certificates.
F1 supplies NumPy/plotting and the Calculus AB baseline supplies arithmetic
with products and floors.

The learning path is:

`C7 module mechanics → C8 neural training → C10 convolutional construction and fine-tuning`.

Sessions 1 and 4 use deterministic CPU-small synthetic data and no downloads.
Sessions 2–3 use the course's locally cached float32 ResNet artifact; their
headers state that separate register explicitly.

## Session map

| Session | Notebook | Time | Covers |
|---|---|---:|---|
| 1 | `lessons/01-convolution-and-feature-maps.ipynb` | 85 min | component-form convolution; detectors and feature maps; output shapes; receptive fields; feature hierarchy |
| 2 | `lessons/02-resnet-reading.ipynb` | 85 min | reproducible ResNet reading; stage shapes; bottleneck anatomy; hand parameter counts |
| 3 | `lessons/03-truncation-and-transfer.ipynb` | 85 min | truncation; prefix freezing; frozen backbone plus fresh head; construction audits |
| 4 | `lessons/04-cnn-training-and-fine-tuning.ipynb` | 90 min | tiny CNN shape ledger; CE/autograd/optimizer lifecycle; ownership, movement, gradient, and buffer certificates; selective fine-tuning |

## Practice map

The manifest is authoritative for all 27 problems.
Sets A and B establish arithmetic, architecture reading, construction, and
audits; Set C integrates them under exam-style constraints.
After Session 4, revisit four 75-minute capstones:

- `p10`: selective prefix freezing plus optimizer-owned training;
- `p24`: commit a hand shape trace, then train the traced convolutional stack;
- `p26`: implement the pure-Python floor-formula tracer, then use it to design
  and train a separate valid stack; and
- `p27`: separate mode, trainability, graph tracking, optimizer ownership,
  parameter motion, and BatchNorm-buffer motion.

These capstones deliberately score the original construction/reasoning
objective separately from the new training objective.

## How to study

1. In Sessions 1–3, write every shape/count before its verification cell.
2. Complete C8 before Session 4; rebuild the five-line training lifecycle
   from memory.
3. In Session 4, maintain four ledgers: shapes, optimizer membership,
   parameters, and buffers. A falling scalar loss alone is never a complete
   certificate.
4. Do `p10`, `p24`, `p26`, and `p27` only after Session 4, then finish with
   `review.ipynb` and redo the linked weak spots.

Total budget: lessons **345 minutes**, practice **875 minutes**, review
**60 minutes**.

In [ ]:
# CPU-small environment check; this overview performs no download or training.
import torch

SEED = 20260804
# torch.device("cpu"): an object naming the CPU; .type is its kind
assert torch.device("cpu").type == "cpu"


<!-- api-card -->
## Code in this unit

A quick reference for the library calls in this unit's code. Each one is explained in a comment (or in the text beside the code) where it first appears.

**New in this unit:**

- `torch.device` — names where tensors live, e.g. `"cpu"`
- `.axis` — `ax.axis("off")` hides the axes and ticks
- `nn.Conv2d` — 2-D convolution layer: `in_channels` to `out_channels` with a `kernel_size` filter
- `.squeeze` — drops the length-1 axes
- `nn.Conv1d` — 1-D convolution layer
- `np.convolve` — 1-D convolution of two arrays (`mode="valid"` keeps full overlaps only)
- `resnet50` — builds ResNet-50, with pretrained weights if `weights=` is given
- `torch.inference_mode` — a `with` block for prediction only: no autograd, faster
- `nn.BatchNorm2d` — batch normalization per channel over `(N, H, W)`
- `torch.is_inference` — True for a tensor made inside `torch.inference_mode`
- `.named_children` — `(name, submodule)` pairs of the top-level submodules
- `getattr` — the attribute with a given name, e.g. `getattr(model, "layer1")`
- `torch.flatten` — flattens the axes from `start_dim` on into one
- `copy.deepcopy` — a fully independent copy, weights and buffers included
- `.conv1` — `model.conv1(x)`: runs only the model's first convolution layer
- `.children` — the top-level submodules, one at a time
- `nn.AdaptiveAvgPool2d` — averages each channel down to a fixed output size, e.g. `1 × 1`
- `nn.MaxPool2d` — the maximum over each window, e.g. `2 × 2` with stride 2
- `.register_forward_hook` — calls a function with each forward pass's input and output
- `id` — a number identifying one object; equal ids mean the same object
- `RuntimeError` — a general error that stops the run with a message
- `nn.Flatten` — a layer flattening `(N, C, H, W)` to `(N, C·H·W)`
- `.modules` — the module itself and every submodule inside it
- `.issubset` — True when every element of one set is in another

**Also used here (met earlier):**

- `np.stack` — join arrays on a new axis (first explained in F4)
- `.eval` — evaluation mode (first explained in C8)
- `.requires_grad_` — turn gradient tracking on/off (first explained in C8)
- `.numel` — number of entries (first explained in C7)
- `nn.Linear` — affine layer `xWᵀ + b` (first explained in C8)
- `.round` — round to given decimals (first explained in F1)
- `any` — True if some item is true (first explained in F7)